# Sesión 12 — Presentaciones del proyecto y cierre del curso

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Cierre

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. **Comunicar** en pocos minutos un proyecto de ML completo: pregunta, datos, pipeline, resultado honesto y límites.
2. **Revisar** su propio proyecto contra las diez trampas del curso.
3. **Comprobar** que su modelo guardado carga y predice fuera del notebook donde se entrenó.
4. **Ubicar** lo aprendido en el pipeline completo y elegir un camino para seguir.

> 💾 **Antes de escribir nada:** menú *Archivo → Guardar una copia en Drive* y trabaje en esa copia; si no, lo que escriba se pierde al cerrar la pestaña. Si Colab avisa que el notebook no es de Google, elija *Ejecutar de todos modos*.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
  ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲
   HOY: todo el pipeline es suyo — su proyecto lo recorre de punta a punta.
```

Este notebook es corto: la sesión es de **ustedes**. Úselo antes de presentar (secciones 2 y 3) y al final (secciones 4 y 5).

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # solo avisos de versiones futuras: los demás (p. ej. InconsistentVersionWarning) importan hoy

REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVOS = {"bank.csv": "datos/bank.csv", "figuras.py": "tools/figuras.py"}

def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo}. Use el Plan B (celda siguiente).")
    return False

for archivo, ruta in ARCHIVOS.items():
    asegurar_datos(archivo, REPO + ruta)
try:
    import figuras as fg
except ModuleNotFoundError:   # Plan B: GitHub no respondió y no llegó figuras.py
    from google.colab import files
    print("⚠️ No llegó figuras.py: súbalo (el profesor lo comparte por el canal del curso).")
    files.upload()
    import figuras as fg
print("Listo:", ", ".join(ARCHIVOS))

In [ ]:
# Plan B: solo actúa si falta algún archivo. Con "Ejecutar todo" no se detiene.
from pathlib import Path
faltan = [a for a in ARCHIVOS if not Path(a).exists()]
if not faltan:
    print("✓ Todo disponible: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ Faltan:", faltan, "→ súbalos")
    files.upload()

In [ ]:
fg.pipeline("todas", modulo=4, subtitulo="HOY: su proyecto recorre el pipeline completo — y lo cuenta en pocos minutos");

## 1. Cómo es la presentación

Todos presentan el mismo día, así que el tiempo se reparte en partes iguales según cuántos seamos. El profesor fija el número; la figura muestra el cronograma de la noche:

In [ ]:
N_PRESENTACIONES = 16        # el profesor lo ajusta al número de inscritos
plan = fg.plan_presentaciones(N_PRESENTACIONES)
coma = lambda v: f"{v:g}".replace(".", ",")
print(f"Cada uno: {coma(plan['charla'])} min de charla + {coma(plan['preguntas'])} de preguntas · colchón de la noche: {coma(plan['colchon'])} min")
fg.cronograma_presentaciones(N_PRESENTACIONES);

Con 13 personas son unos 4,5 minutos; con 20, unos 3. Siempre queda un colchón de al menos 3 minutos para los cambios de pantalla. Es poco, y es a propósito: **contar un proyecto en pocos minutos es una habilidad** que se usa con jefes, clientes y jurados.

**Cuatro láminas, ni una más** (una quinta opcional con la figura SHAP):

| # | Lámina | Qué tiene que quedar claro |
|---|---|---|
| 1 | **La pregunta** | Qué quiere predecir o descubrir, para quién sería útil y con qué métrica se mide el éxito (y por qué esa) |
| 2 | **Los datos y el pipeline** | De dónde salen, cuántas filas; el pipeline en una línea; **qué trampa evitó** (fuga, escala, desbalance…) |
| 3 | **El resultado honesto** | Línea base → E2 → modelo final, con validación cruzada (±) y **el número de prueba**; cuánto ganó afinar |
| 4 | **Lo que aprendió y lo que no se puede concluir** | Un error que cometió y cómo lo detectó; qué **no** dice su modelo (causalidad, otras poblaciones, otras fechas) |

**Reglas de la noche:** pantalla compartida lista antes de su turno · el cronómetro es del curso: al minuto se corta con amabilidad y se pasa a preguntas · las preguntas las hace primero un compañero (el siguiente en la lista) y luego el profesor.

## 2. Antes de presentar: las diez trampas del curso

Durante doce sesiones nos equivocamos a propósito. Ninguna de estas trampas produjo un error de Python: **todas produjeron un número bonito**.

In [ ]:
fg.trampas_del_curso();

Revise su proyecto contra cada una. Cambie a `True` solo las frases que **ya verificó** en su notebook (no las que "seguro no tiene"):

In [ ]:
revisado = {
    "S2  Verifiqué que ninguna variable refleja cómo se REGISTRA el dato en vez del fenómeno (año, fuente, formato)": False,
    "S3  Verifiqué que las estadísticas (promedios, codificaciones) se calculan solo con el entrenamiento": False,
    "S4  Verifiqué que ninguna X es parte de la y ni se calcula a partir de ella": False,
    "S5  Verifiqué que comparé mi modelo contra la línea base que siempre dice lo mismo": False,
    "S6  Verifiqué que todas las columnas se conocen ANTES de la y": False,
    "S7  Verifiqué que escalé antes de usar distancias (k-NN, K-means, PCA)": False,
    "S8  Verifiqué que no leo tamaños ni distancias en un t-SNE": False,
    "S9  Verifiqué que no metí un resumen junto a sus propias partes": False,
    "S10 Verifiqué que todo lo que aprende de los datos está DENTRO del Pipeline": False,
    "S11 Verifiqué que el número que reporto es el de prueba y no el best_score_": False,
}
hechos = sum(revisado.values())
print(f"Verificadas: {hechos} de {len(revisado)}")
for k, v in revisado.items():
    if not v:
        print("   pendiente →", k)

Si alguna trampa le aparece **ahora**, no la esconda: cuéntela en la lámina 4. Encontrar un error propio y explicarlo vale más en la rúbrica que un número alto sin explicación.

## 3. Prueba de humo: ¿su modelo funciona fuera de su notebook?

En E4 guardó el `Pipeline` final con `joblib`. Un modelo que solo funciona dentro del notebook donde se entrenó no le sirve a nadie más. Esta celda hace lo que haría quien lo reciba: **cargarlo en un Colab limpio** y pedirle una predicción con datos crudos.

1. Suba su archivo `.joblib` (ícono de carpeta a la izquierda → subir) y un CSV con **2 o 3 filas** de su dataset, **sin la columna *y***.
2. Cambie los nombres en la celda y ejecútela.

In [ ]:
import joblib
from pathlib import Path

MI_MODELO = "mi_modelo.joblib"        # ← su archivo de E4
MIS_FILAS = "filas_nuevas.csv"        # ← 2 o 3 filas de su dataset, SIN la columna y

if Path(MI_MODELO).exists() and Path(MIS_FILAS).exists():
    modelo = joblib.load(MI_MODELO)
    filas = pd.read_csv(MIS_FILAS)
    print("✓ El modelo cargó:", type(modelo).__name__)
    print(modelo.predict(filas))
    if hasattr(modelo, "predict_proba"):
        print(modelo.predict_proba(filas).round(3))
else:
    print("Aún no subió su modelo y sus filas: abajo va la demostración con el modelo del banco (S11).")

Si no ha subido nada, la celda siguiente hace la demostración con un modelo de Bank Marketing: lo entrena, lo guarda, **lo borra de la memoria** y lo vuelve a cargar desde el archivo, como si fuera otra persona en otro computador.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

bank = pd.read_csv("bank.csv")
crudas = bank.drop(columns=["deposit", "duration"])
y = (bank["deposit"] == "yes").astype(int)
categoricas = crudas.select_dtypes(exclude="number").columns.tolist()
demo = make_pipeline(make_column_transformer((OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas),
                                             remainder="passthrough"),
                     HistGradientBoostingClassifier(random_state=42)).fit(crudas, y)   # ya evaluado en S11: el modelo
                                                                                       # que se entrega se reentrena con todas las filas
joblib.dump(demo, "demo_bank.joblib")
del demo                                   # ya no existe en memoria: solo queda el archivo

cargado = joblib.load("demo_bank.joblib")
nuevos = crudas.sample(3, random_state=1)  # filas crudas, tal como llegarían
print(nuevos[["age", "job", "balance", "contact", "poutcome"]])
print("probabilidad de 'sí':", cargado.predict_proba(nuevos)[:, 1].round(2))

**Errores típicos de la prueba de humo** (y qué significan):
- `KeyError` o `ValueError: columns are missing` → el `Pipeline` espera columnas que sus filas nuevas no traen, o el preprocesamiento quedó **fuera** del `Pipeline` (se hizo `get_dummies` antes de entrenar).
- `ValueError: Found unknown categories` → falta `handle_unknown="ignore"` en el `OneHotEncoder`.
- `ModuleNotFoundError` → el modelo usa una librería que no está instalada aquí (por ejemplo, LightGBM): hay que instalarla antes de cargar.

Si su modelo pasa esta prueba, ya tiene algo que **otra persona** puede usar.

## 4. El curso en una figura

Doce sesiones, cinco etapas, cuatro módulos. Desde la Sesión 1 corrimos el pipeline completo; cada sesión profundizó una o dos etapas:

In [ ]:
fg.mapa_del_curso();

**Dentro del `.fit()` — los modelos del curso:**

| | Árbol | Lineal | Logística | k-NN | Bosque | Boosting | K-means | DBSCAN | PCA | t-SNE |
|---|---|---|---|---|---|---|---|---|---|---|
| ¿Usa *y*? | sí | sí | sí | sí | sí | sí | no | no | no | no |
| Qué minimiza | Gini | residuales² | log-loss | — | Gini por árbol | log-loss (suma de árboles) | inercia | — | − varianza | KL |
| Cómo lo busca | voraz | cerrada o gradiente | iterativo | guarda y vota | árboles en paralelo | árboles en fila (gradiente) | Lloyd | encadena | álgebra lineal | gradiente |
| Perilla clave | `max_depth` | `alpha` | `C` | `k` | `n_estimators` | `learning_rate` | `k` | `eps` | `n_components` | `perplexity` |

Y la búsqueda de hiperparámetros (grilla, azar, Optuna) es **otra optimización encima de todas**: minimiza el error de validación, sin gradiente.

**Cinco hábitos que valen más que cualquier algoritmo:**
1. **Separar antes de todo** — la prueba se aparta primero y se mira una sola vez (S3, S10, S11).
2. **Línea base siempre** — un número sin comparación no dice nada (S1, S5).
3. **El número con su ±** — una sola partición puede engañar (S10).
4. **Preguntar de dónde sale cada columna** — ¿la tendría antes de conocer la *y*? (S3, S4, S6).
5. **Desconfiar del número bonito** — diez veces en este curso fue el síntoma de un error.

## 5. ¿Y ahora qué?

In [ ]:
fg.que_sigue();

**Para su proyecto, después del curso:** el notebook de E4 es un buen punto de partida para un portafolio. Súbalo a GitHub con un `README.md` corto (la pregunta, los datos, el resultado honesto y cómo correrlo). Un proyecto con una lectura honesta de sus límites dice más de usted que un número alto.

## Gracias

Gracias por estas doce sesiones. La encuesta de evaluación del curso es corta y nos ayuda a mejorarlo para la próxima cohorte: **[enlace de la encuesta de Univalle]**.

El repositorio del curso queda publicado: los notebooks, las figuras y los datos siguen disponibles para que vuelva a ellos cuando los necesite.